In [ ]:
# CatBoost is not preinstalled on every Kaggle image
!pip install -q catboost

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
print("TensorFlow", tf.__version__)

## 1. Import the dataset

This cell finds the CSV on Kaggle **or** on your PC, so the same notebook works in both places.

In [ ]:
CSV_NAME = "water_quality_training_samples.csv"

def find_dataset():
    hits = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working"), Path("."), Path("..")]:
        if root.exists():
            hits.extend(root.rglob(CSV_NAME))
    local = Path(r"C:/Users/asus/OneDrive/Desktop/Aqamind/Water - DataSet") / CSV_NAME
    if local.exists():
        hits.append(local)
    if not hits:
        raise FileNotFoundError(
            f"Could not find {CSV_NAME}. On Kaggle: Add Input → your dataset. "
            "On PC: keep the CSV next to this notebook."
        )
    return hits[0]

DATA_PATH = find_dataset()
print("Loaded from:", DATA_PATH)

df = pd.read_csv(DATA_PATH)
print("Rows:", len(df), " | Columns:", len(df.columns))
df.head()

In [ ]:
print(df.info())
print("\nMissing values:\n", df.isna().sum())
print("\nQuality counts:\n", df["Quality"].value_counts())
print("\nKind counts:\n", df["Kind"].value_counts())

## 2. Charts and heatmap

These plots show class balance, how each parameter looks, and which readings move together.

In [ ]:
QUALITY_ORDER = ["excellent", "good", "watch", "critical"]
QUALITY_COLORS = {
    "excellent": "#0F766E",
    "good": "#65A30D",
    "watch": "#D97706",
    "critical": "#DC2626",
}
PARAMS = ["pH", "Temperature", "Ammonia", "Nitrite", "Nitrate", "DissolvedO2"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x="Quality", order=QUALITY_ORDER, palette=QUALITY_COLORS, ax=axes[0])
axes[0].set_title("Quality class distribution")
sns.countplot(data=df, x="Kind", palette=["#0F766E", "#65A30D"], ax=axes[1])
axes[1].set_title("Fish vs Plant samples")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), PARAMS):
    sns.histplot(df[col], kde=True, color="#0F766E", ax=ax)
    ax.set_title(col)
plt.suptitle("Water parameter distributions", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), PARAMS):
    sns.boxplot(
        data=df,
        x="Quality",
        y=col,
        order=QUALITY_ORDER,
        palette=QUALITY_COLORS,
        ax=ax,
    )
    ax.set_title(f"{col} by quality")
    ax.tick_params(axis="x", rotation=20)
plt.suptitle("How each parameter changes when water gets worse", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
corr = df[PARAMS].corr()
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.6,
)
plt.title("Correlation heatmap — 6 water parameters")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 4.5))
class_means = df.groupby("Quality")[PARAMS].mean().reindex(QUALITY_ORDER)
sns.heatmap(class_means, annot=True, fmt=".2f", cmap="YlOrRd", linewidths=0.6)
plt.title("Mean parameter values by quality class")
plt.ylabel("")
plt.tight_layout()
plt.show()

class_means

In [ ]:
FEATURES = [
    "pH",
    "Temperature",
    "Ammonia",
    "Nitrite",
    "Nitrate",
    "DissolvedO2",
    "Min pH",
    "Max pH",
    "Min Temp (C)",
    "Max Temp (C)",
    "Max Safe Ammonia (ppm)",
    "Max Safe Nitrite (ppm)",
    "Max Safe Nitrate (ppm)",
    "Min Dissolved O2 (mg/L)",
]
TARGET = "Quality"

X = df[FEATURES].copy()
y_text = df[TARGET].copy()

label_encoder = LabelEncoder()
label_encoder.fit(QUALITY_ORDER)
y = label_encoder.transform(y_text)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Classes:", list(label_encoder.classes_))

In [ ]:
results = []
predictions = {}

def score_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    results.append({"Model": name, "Accuracy": round(acc, 4), "F1 macro": round(f1, 4)})
    predictions[name] = y_pred
    print(f"\n===== {name} =====")
    print(f"Accuracy : {acc * 100:.2f}%")
    print(f"F1 macro : {f1 * 100:.2f}%")
    print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))
    return acc

## 4. Logistic Regression

In [ ]:
log_reg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
log_reg.fit(X_train_s, y_train)
pred_lr = log_reg.predict(X_test_s)
score_model("Logistic Regression", y_test, pred_lr)

## 5. Decision Tree

In [ ]:
tree = DecisionTreeClassifier(
    max_depth=12,
    min_samples_leaf=4,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
tree.fit(X_train, y_train)
pred_tree = tree.predict(X_test)
score_model("Decision Tree", y_test, pred_tree)

## 6. Artificial Neural Network

In [ ]:
n_features = X_train_s.shape[1]
n_classes = len(label_encoder.classes_)

ann = Sequential(
    [
        Dense(64, activation="relu", input_shape=(n_features,)),
        BatchNormalization(),
        Dropout(0.25),
        Dense(32, activation="relu"),
        Dropout(0.20),
        Dense(16, activation="relu"),
        Dense(n_classes, activation="softmax"),
    ]
)
ann.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
ann.summary()

In [ ]:
history = ann.fit(
    X_train_s,
    y_train,
    validation_split=0.15,
    epochs=80,
    batch_size=32,
    verbose=1,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    ],
)

pred_ann = np.argmax(ann.predict(X_test_s, verbose=0), axis=1)
score_model("Artificial Neural Network", y_test, pred_ann)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("ANN loss")
axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("ANN accuracy")
axes[1].legend()
plt.tight_layout()
plt.show()

## 7. CatBoost

In [ ]:
cat = CatBoostClassifier(
    iterations=400,
    depth=6,
    learning_rate=0.08,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=RANDOM_STATE,
    verbose=100,
)
cat.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)
pred_cat = cat.predict(X_test).ravel().astype(int)
score_model("CatBoost", y_test, pred_cat)

## 8. Compare models and keep the best

In [ ]:
scoreboard = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
scoreboard.index = scoreboard.index + 1
display(scoreboard)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = scoreboard.melt(id_vars="Model", value_vars=["Accuracy", "F1 macro"], var_name="Metric", value_name="Score")
sns.barplot(data=plot_df, x="Model", y="Score", hue="Metric", ax=ax)
ax.set_ylim(0, 1.05)
ax.set_title("Model comparison")
plt.xticks(rotation=15)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=8)
plt.tight_layout()
plt.show()

best_name = scoreboard.iloc[0]["Model"]
best_acc = scoreboard.iloc[0]["Accuracy"]
print(f"\nBEST MODEL: {best_name}   accuracy = {best_acc * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, name in zip(axes.ravel(), predictions):
    cm = confusion_matrix(y_test, predictions[name], labels=range(len(QUALITY_ORDER)))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=QUALITY_ORDER,
        yticklabels=QUALITY_ORDER,
        ax=ax,
    )
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.suptitle("Confusion matrices", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
if best_name == "CatBoost":
    importances = pd.Series(cat.get_feature_importance(), index=FEATURES).sort_values()
elif best_name == "Decision Tree":
    importances = pd.Series(tree.feature_importances_, index=FEATURES).sort_values()
elif best_name == "Logistic Regression":
    importances = pd.Series(np.abs(log_reg.coef_).mean(axis=0), index=FEATURES).sort_values()
else:
    importances = None

if importances is not None:
    plt.figure(figsize=(8, 6))
    importances.plot(kind="barh", color="#0F766E")
    plt.title(f"Feature importance — {best_name}")
    plt.tight_layout()
    plt.show()
else:
    print("ANN has no simple feature-importance chart. Use the confusion matrix above.")

## 9. Save the best model

On Kaggle the files appear under **Output**. Download them and send them back so we can plug the winner into the Water page.

In [ ]:
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT.mkdir(exist_ok=True)

bundle = {
    "best_model_name": best_name,
    "accuracy": float(best_acc),
    "feature_names": FEATURES,
    "labels": list(label_encoder.classes_),
    "scaler": scaler,
    "label_encoder": label_encoder,
    "scoreboard": scoreboard,
}

if best_name == "Logistic Regression":
    bundle["model"] = log_reg
    joblib.dump(bundle, OUT / "water_quality_best_model.pkl")
elif best_name == "Decision Tree":
    bundle["model"] = tree
    joblib.dump(bundle, OUT / "water_quality_best_model.pkl")
elif best_name == "CatBoost":
    cat.save_model(str(OUT / "water_quality_catboost.cbm"))
    joblib.dump(bundle, OUT / "water_quality_best_model.pkl")
else:
    ann.save(OUT / "water_quality_ann.keras")
    joblib.dump(bundle, OUT / "water_quality_best_model.pkl")

scoreboard.to_csv(OUT / "model_scoreboard.csv", index=False)
print("Saved files in", OUT.resolve())
print(" - water_quality_best_model.pkl")
print(" - model_scoreboard.csv")
if best_name == "CatBoost":
    print(" - water_quality_catboost.cbm")
if best_name == "Artificial Neural Network":
    print(" - water_quality_ann.keras")
print("\nDownload these from the Kaggle Output panel and send them back.")